# Lab 3: High-Accuracy Transfer Learning Model

In this lab, we build a complete, production-quality transfer learning pipeline using MobileNetV2 on the full CIFAR-10 dataset (10 classes). We incorporate data augmentation, a structured training pipeline, comprehensive evaluation, and model saving.

## Learning Objectives
- Build a complete transfer learning pipeline with data augmentation
- Train on the full CIFAR-10 dataset (10 classes)
- Evaluate with confusion matrices and classification reports
- Visualize correct and incorrect predictions
- Save and load trained models
- Build a Gradio interface with top-5 predictions

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision gradio python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

## 1. Setup and Installation

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Load the Full CIFAR-10 Dataset

In [ ]:
# Load the full CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize pixel values to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train = y_train.flatten()
y_test = y_test.flatten()

class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]
num_classes = len(class_names)

print(f"Training set: {x_train.shape}, Labels: {y_train.shape}")
print(f"Test set: {x_test.shape}, Labels: {y_test.shape}")
print(f"Number of classes: {num_classes}")
print(f"\nClass distribution (train):")
for i, name in enumerate(class_names):
    print(f"  {name}: {np.sum(y_train == i)}")

In [ ]:
# Visualize samples from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    idx = np.where(y_train == i)[0][0]
    ax.imshow(x_train[idx])
    ax.set_title(class_names[i], fontsize=11)
    ax.axis("off")
plt.suptitle("CIFAR-10: One Sample Per Class", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Build the Model with Data Augmentation

We create a complete pipeline with augmentation layers, pre-trained MobileNetV2 base, and a custom multi-class classification head.

In [ ]:
# Define data augmentation layers
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),
], name="data_augmentation")

# Load the pre-trained MobileNetV2 base model
base_model = keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(96, 96, 3)
)

# Freeze base model initially
base_model.trainable = False

# Build the complete model
inputs = keras.Input(shape=(32, 32, 3))

# Resize to 96x96 for MobileNetV2
x = keras.layers.Resizing(96, 96)(inputs)

# Apply data augmentation (only during training)
x = data_augmentation(x)

# Apply MobileNetV2 preprocessing
x = keras.applications.mobilenet_v2.preprocess_input(x)

# Pass through frozen base model
x = base_model(x, training=False)

# Custom classification head
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.3)(x)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="high_accuracy_transfer_model")
model.summary()

## 4. Phase 1: Train the Classification Head

In [ ]:
# Compile for Phase 1
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("=" * 50)
print("Phase 1: Training Classification Head (frozen base)")
print("=" * 50)

history_phase1 = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

phase1_loss, phase1_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nPhase 1 Test Accuracy: {phase1_acc:.4f}")

## 5. Phase 2: Fine-Tune Top Layers

In [ ]:
# Unfreeze the top layers of the base model
base_model.trainable = True

# Freeze all layers except the last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_layers = sum(1 for l in base_model.layers if l.trainable)
print(f"Unfrozen layers in base model: {trainable_layers}")

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
)
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1
)

print("\n" + "=" * 50)
print("Phase 2: Fine-Tuning Top Layers")
print("=" * 50)

history_phase2 = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

phase2_loss, phase2_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nPhase 2 Test Accuracy: {phase2_acc:.4f}")
print(f"Improvement: {(phase2_acc - phase1_acc) * 100:.2f} percentage points")

## 6. Plot Learning Curves

In [ ]:
# Combined learning curves
all_train_acc = history_phase1.history["accuracy"] + history_phase2.history["accuracy"]
all_val_acc = history_phase1.history["val_accuracy"] + history_phase2.history["val_accuracy"]
all_train_loss = history_phase1.history["loss"] + history_phase2.history["loss"]
all_val_loss = history_phase1.history["val_loss"] + history_phase2.history["val_loss"]

phase1_end = len(history_phase1.history["accuracy"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(all_train_acc, label="Train Accuracy", marker="o", markersize=4)
ax1.plot(all_val_acc, label="Val Accuracy", marker="s", markersize=4)
ax1.axvline(x=phase1_end - 0.5, color="red", linestyle="--", label="Start Fine-Tuning")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.set_title("Accuracy: Feature Extraction -> Fine-Tuning")
ax1.legend()
ax1.grid(True)

ax2.plot(all_train_loss, label="Train Loss", marker="o", markersize=4)
ax2.plot(all_val_loss, label="Val Loss", marker="s", markersize=4)
ax2.axvline(x=phase1_end - 0.5, color="red", linestyle="--", label="Start Fine-Tuning")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.set_title("Loss: Feature Extraction -> Fine-Tuning")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 7. Comprehensive Evaluation

We evaluate the model using a confusion matrix and classification report to understand per-class performance.

In [ ]:
# Get predictions on the test set
y_pred_probs = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Overall accuracy
accuracy = np.mean(y_pred == y_test)
print(f"Test Accuracy: {accuracy:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix", fontsize=14)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

## 8. Visualize Correct and Incorrect Predictions

In [ ]:
# Find correct and incorrect predictions
correct_mask = y_pred == y_test
incorrect_mask = ~correct_mask

correct_indices = np.where(correct_mask)[0]
incorrect_indices = np.where(incorrect_mask)[0]

print(f"Correct predictions: {len(correct_indices)} ({len(correct_indices)/len(y_test)*100:.1f}%)")
print(f"Incorrect predictions: {len(incorrect_indices)} ({len(incorrect_indices)/len(y_test)*100:.1f}%)")

In [ ]:
# Visualize correct predictions
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Correct Predictions (green)", fontsize=14)
np.random.seed(42)
sample_correct = np.random.choice(correct_indices, 10, replace=False)

for i, ax in enumerate(axes.flat):
    idx = sample_correct[i]
    ax.imshow(x_test[idx])
    conf = y_pred_probs[idx][y_pred[idx]]
    ax.set_title(f"{class_names[y_pred[idx]]}\nConf: {conf:.2f}", color="green", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Visualize incorrect predictions
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Incorrect Predictions (red = predicted, green = true)", fontsize=14)
num_show = min(10, len(incorrect_indices))
sample_incorrect = np.random.choice(incorrect_indices, num_show, replace=False)

for i, ax in enumerate(axes.flat):
    if i < num_show:
        idx = sample_incorrect[i]
        ax.imshow(x_test[idx])
        conf = y_pred_probs[idx][y_pred[idx]]
        ax.set_title(f"Pred: {class_names[y_pred[idx]]} ({conf:.2f})\nTrue: {class_names[y_test[idx]]}",
                     color="red", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Save the Model

In [ ]:
# Save the model
save_path = "cifar10_mobilenetv2_finetuned.keras"
model.save(save_path)
print(f"Model saved to: {save_path}")

# Verify by loading and evaluating
loaded_model = keras.models.load_model(save_path)
loaded_loss, loaded_acc = loaded_model.evaluate(x_test, y_test, verbose=0)
print(f"Loaded model test accuracy: {loaded_acc:.4f}")

## 10. Gradio Interface

Interactive interface showing top-5 predictions with confidence scores.

In [ ]:
import gradio as gr

def classify_image(image):
    """Classify an uploaded image into one of 10 CIFAR-10 classes."""
    if image is None:
        return {name: 0.0 for name in class_names}

    # Preprocess
    img = np.array(image)
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    if img.shape[-1] == 4:
        img = img[:, :, :3]

    from PIL import Image
    img_pil = Image.fromarray(img.astype(np.uint8)).resize((32, 32))
    img_array = np.array(img_pil).astype("float32") / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    predictions = model.predict(img_array, verbose=0)[0]

    # Return top-5 predictions as a dictionary
    top5_indices = np.argsort(predictions)[::-1][:5]
    result = {class_names[i]: float(predictions[i]) for i in top5_indices}
    return result

demo = gr.Interface(
    fn=classify_image,
    inputs=gr.Image(type="numpy", label="Upload an Image"),
    outputs=gr.Label(num_top_classes=5, label="Top-5 Predictions"),
    title="CIFAR-10 High-Accuracy Transfer Learning Classifier",
    description="Upload an image to classify it into one of 10 CIFAR-10 categories using a fine-tuned MobileNetV2 model. Shows top-5 predictions with confidence.",
    examples=None,
    flagging_mode="never"
)

demo.launch(share=False)